# Hog Production Regression Models

Two OLS models explaining annual U.S. hog production (billions of pounds) using price and input cost data.
- **Model 1:** Lagged OLS — all regressors shifted one year to reflect production planning lags
- **Model 2:** Log-Log OLS — coefficients interpreted as elasticities

## Load Libraries and Data

In [11]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv('data/pork_supply_data.csv')

print(f"Observations: {len(df)}  |  Years: {df['year'].min()}–{df['year'].max()}")
df.head()

Observations: 37  |  Years: 1988–2024


,year,hog_production_lb,hog_prices_lb,corn_prices_bu,soybean_meal_price_lb
0,2024,41.604752,0.528861,3.498238,0.123664
1,2023,42.424619,0.526622,3.864727,0.163130
2,2022,40.674251,0.645579,5.783684,0.199825
3,2021,39.992279,0.654263,5.730767,0.210037
4,2020,40.428021,0.471000,4.530000,0.196155


## Model 1: Lagged OLS

Hog producers make supply decisions based on last year's conditions. We use hog prices prices observed last year, feed costs last year, and previous year hog production. Lagging all regressors by one year captures this planning horizon.

$$\text{production}_t = \beta_0 + \beta_1 \text{production}_{t-1} + \beta_2 \text{hog price}_{t-1} + \beta_3 \text{corn price}_{t-1} + \beta_4 \text{soy meal price}_{t-1} + \varepsilon$$

In [20]:
# Create one-year lags for all variables
lag_cols = ['hog_production_lb', 'hog_prices_lb', 'corn_prices_bu', 'soybean_meal_price_lb']
for col in lag_cols:
    df[f'{col}_lag1'] = df[col].shift(1)  # shift() moves values down by n rows

df_lag = df.dropna()  # first row has no lag values — drop it

# Sort first to ensure correct ordering
df_lag = df_lag.sort_values('year').reset_index(drop=True)

# Create sequential time trend starting at 1
df_lag['t'] = range(1, len(df_lag) + 1)

model1 = smf.ols(
    'hog_production_lb ~ hog_production_lb_lag1 + hog_prices_lb_lag1 + corn_prices_bu_lag1 + soybean_meal_price_lb_lag1 + t',
    data=df_lag
).fit()

print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:      hog_production_lb   R-squared:                       0.977
Model:                            OLS   Adj. R-squared:                  0.973
Method:                 Least Squares   F-statistic:                     250.0
Date:                Sun, 12 Apr 2026   Prob (F-statistic):           1.69e-23
Time:                        17:51:20   Log-Likelihood:                -50.071
No. Observations:                  36   AIC:                             112.1
Df Residuals:                      30   BIC:                             121.6
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

### Model 1 — Coefficient Interpretation

Coefficients are in **levels** (lbs per capita per unit change in regressor):

| Variable | Expected sign | Interpretation |
|---|---|---|
| `hog_production_lb_lag1` | (+) | Higher production last year suggests that output will be higher this year |
| `hog_prices_lb_lag1` | (+) | Higher prices last year incentivize more production this year |
| `corn_prices_bu_lag1` | (−) | Corn is the primary feed input; higher cost → lower supply |
| `soybean_meal_price_lb_lag1` | (−) | Soy meal is a primary protein supplement; same logic |

## Model 2: Log-Log OLS

Taking logs of both sides transforms the model so each coefficient is an **elasticity**. An elasticity indicates the percent change in production associated with a 1% change in the regressor.

$$\ln(\text{production}) = \beta_0 + \beta_1 \ln(\text{hog price}) + \beta_2 \ln(\text{corn price}) + \beta_3 \ln(\text{soy meal price}) + \varepsilon$$

Statsmodels formula API supports `np.log()` inline, so we do not need to create new columns.

In [19]:
model2 = smf.ols(
    'np.log(hog_production_lb) ~ np.log(hog_prices_lb_lag1) + np.log(corn_prices_bu_lag1) + np.log(soybean_meal_price_lb_lag1) + np.log(hog_production_lb_lag1) + t',
    data=df_lag
).fit()

print(model2.summary())

                                OLS Regression Results                               
Dep. Variable:     np.log(hog_production_lb)   R-squared:                       0.979
Model:                                   OLS   Adj. R-squared:                  0.975
Method:                        Least Squares   F-statistic:                     278.6
Date:                       Sun, 12 Apr 2026   Prob (F-statistic):           3.46e-24
Time:                               17:50:59   Log-Likelihood:                 75.239
No. Observations:                         36   AIC:                            -138.5
Df Residuals:                             30   BIC:                            -129.0
Df Model:                                  5                                         
Covariance Type:                   nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------

### Model 2 — Coefficient Interpretation

Coefficients are **elasticities**: a coefficient of 0.9802 on `log(hog_production_lb_lag1)` means a 10% increase in last year's hog production is associated with a 9.802% increase in production.

Elasticities are scale-free and directly comparable across variables.

## Model Diagnostics

In [4]:
# Key fit statistics for the lagged levels model
print(f"{'Metric':<25} {'Model 1 (Lagged Levels)':>24}")
print('-' * 50)
print(f"{'R-squared':<25} {model1.rsquared:>24.3f}")
print(f"{'Adj. R-squared':<25} {model1.rsquared_adj:>24.3f}")
print(f"{'AIC':<25} {model1.aic:>24.1f}")
print(f"{'BIC':<25} {model1.bic:>24.1f}")
print(f"{'F-stat (p-value)':<25} {model1.f_pvalue:>24.4f}")
print(f"{'Observations':<25} {int(model1.nobs):>24}")

Metric                     Model 1 (Lagged Levels)
--------------------------------------------------
R-squared                                    0.975
Adj. R-squared                               0.972
AIC                                          111.9
BIC                                          119.9
F-stat (p-value)                            0.0000
Observations                                    36
